# Phrase similarity

Three simple ways to compare phrases, using only the standard library:

1. **Character similarity** (`difflib`) - good for typos
2. **Token Jaccard** - good for reordered words
3. **Bag-of-words cosine** - like Jaccard, but counts repeated words

In [1]:
import math
import re
from collections import Counter
from difflib import SequenceMatcher


def normalize(text: str) -> str:
    """Lowercase, drop punctuation, collapse whitespace."""
    text = re.sub(r"[^a-z0-9\s]", "", text.lower())
    return re.sub(r"\s+", " ", text).strip()


def tokens(text: str) -> list[str]:
    return normalize(text).split()


def char_similarity(a: str, b: str) -> float:
    return SequenceMatcher(None, normalize(a), normalize(b)).ratio()


def jaccard(a: str, b: str) -> float:
    sa, sb = set(tokens(a)), set(tokens(b))
    return len(sa & sb) / len(sa | sb) if sa | sb else 0.0


def cosine(a: str, b: str) -> float:
    ca, cb = Counter(tokens(a)), Counter(tokens(b))
    dot = sum(ca[w] * cb[w] for w in ca)
    norm = math.sqrt(sum(v * v for v in ca.values())) * math.sqrt(sum(v * v for v in cb.values()))
    return dot / norm if norm else 0.0

## Compare some pairs

In [2]:
pairs = [
    ("Men's Cycling Gloves", "mens cycling gloves"),            # punctuation / case
    ("Cycling Gloves Men", "Men Cycling Gloves"),                # reordered
    ("Winter Cycling Gloves", "Winter Cyclng Glovs"),            # typos
    ("Waterproof Thermal Gloves", "Thermal Waterproof Glove"),   # reorder + singular
    ("Cycling Gloves", "Running Socks"),                         # unrelated
]

print(f"{'phrase_a':28}{'phrase_b':28}{'char':>6}{'jacc':>6}{'cos':>6}")
for a, b in pairs:
    print(f"{a:28}{b:28}{char_similarity(a, b):6.2f}{jaccard(a, b):6.2f}{cosine(a, b):6.2f}")

phrase_a                    phrase_b                      char  jacc   cos
Men's Cycling Gloves        mens cycling gloves           1.00  1.00  1.00
Cycling Gloves Men          Men Cycling Gloves            0.78  1.00  1.00
Winter Cycling Gloves       Winter Cyclng Glovs           0.95  0.20  0.33
Waterproof Thermal Gloves   Thermal Waterproof Glove      0.65  0.50  0.67
Cycling Gloves              Running Socks                 0.44  0.00  0.00


Notice how each method behaves:

- **Reordered words** score low on character similarity but 1.0 on Jaccard/cosine.
- **Typos** score high on character similarity but low on token methods (the words no longer match exactly).
- **Singular vs plural** (`Gloves` / `Glove`) breaks token matching unless you stem/lemmatize first.

## Find the best match from a list

This is the core of word normalization: map a messy phrase to a canonical one.

In [3]:
canonical = ["Cycling Gloves", "Cycling Socks", "Running Shoes", "Thermal Jacket"]
messy = ["cyclng glovs", "Gloves, Cycling", "thermal jackets", "sock for cycling"]


def best_match(phrase: str, candidates: list[str]) -> tuple[str, float]:
    # Blend both signals so typos and reordering are both handled
    scored = [(c, (char_similarity(phrase, c) + cosine(phrase, c)) / 2) for c in candidates]
    return max(scored, key=lambda x: x[1])


for m in messy:
    match, score = best_match(m, canonical)
    print(f"{m:20} -> {match:16} ({score:.2f})")

cyclng glovs         -> Cycling Gloves   (0.46)
Gloves, Cycling      -> Cycling Gloves   (0.75)
thermal jackets      -> Thermal Jacket   (0.73)
sock for cycling     -> Cycling Socks    (0.45)


## The same thing with `rapidfuzz`

`rapidfuzz` is fast and has scorers built for this. Scores are 0-100.

- `ratio`: plain character similarity (like `difflib`)
- `token_sort_ratio`: sorts words first, so reordering doesn't matter
- `token_set_ratio`: ignores extra words, so a subset of words still matches
- `WRatio`: picks a sensible blend automatically

In [4]:
from rapidfuzz import fuzz, process, utils

print(f"{'phrase_a':28}{'phrase_b':28}{'ratio':>7}{'sort':>7}{'set':>7}{'WRatio':>8}")
for a, b in pairs:
    print(
        f"{a:28}{b:28}"
        f"{fuzz.ratio(a, b, processor=utils.default_process):7.0f}"
        f"{fuzz.token_sort_ratio(a, b, processor=utils.default_process):7.0f}"
        f"{fuzz.token_set_ratio(a, b, processor=utils.default_process):7.0f}"
        f"{fuzz.WRatio(a, b, processor=utils.default_process):8.0f}"
    )

phrase_a                    phrase_b                      ratio   sort    set  WRatio
Men's Cycling Gloves        mens cycling gloves              97     97     97      97
Cycling Gloves Men          Men Cycling Gloves               78    100    100      95
Winter Cycling Gloves       Winter Cyclng Glovs              95     95     95      95
Waterproof Thermal Gloves   Thermal Waterproof Glove         65     98     98      93
Cycling Gloves              Running Socks                    44     44     44      44


### Best match from the canonical list with `process.extractOne`

In [5]:
for m in messy:
    match, score, _ = process.extractOne(
        m, canonical, scorer=fuzz.WRatio, processor=utils.default_process
    )
    print(f"{m:20} -> {match:16} ({score:.0f})")

cyclng glovs         -> Cycling Gloves   (92)
Gloves, Cycling      -> Cycling Gloves   (95)
thermal jackets      -> Thermal Jacket   (97)
sock for cycling     -> Cycling Socks    (79)
